# 第13章：人机协作（Human-in-the-Loop）

## 概念

**人机协作** = 在 LLM 工作流程中引入人类决策，确保关键操作可控

```
LLM 分析 → 生成方案 → 需要人工确认？ → 是 → 等待人类审批 → 执行
                                    ↓ 否
                                  直接执行
```

## 人机协作模式

| 模式 | 说明 | 示例 |
|------|------|------|
| **审批确认** | 关键操作前需人类批准 | 发送邮件前确认 |
| **内容审核** | 人类审核 LLM 输出 | 审核生成的文章 |
| **升级处理** | 复杂问题转交人类 | 客服转人工 |

## 代码演示

使用 LangChain 实现三种人机协作模式

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
load_dotenv()

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0
)

print("MiMo 模型初始化完成！")

MiMo 模型初始化完成！


## 模式1：审批确认

关键操作前需要人类批准

In [3]:
# 模拟需要审批的操作
def send_email(to: str, subject: str, body: str) -> dict:
    """模拟发送邮件"""
    return {"status": "sent", "to": to, "subject": subject}

def delete_account(user_id: str) -> dict:
    """模拟删除账户"""
    return {"status": "deleted", "user_id": user_id}

# 带审批的操作执行器
def execute_with_approval(action_name: str, action_func, auto_approve: bool = False, **kwargs):
    """执行操作前需要审批"""
    print(f"\n[系统] 准备执行: {action_name}")
    print(f"[系统] 参数: {kwargs}")
    
    if auto_approve:
        print("[系统] 自动审批通过")
        return action_func(**kwargs)
    
    # 模拟人工审批（在实际系统中会等待用户输入）
    approval = input(f"[审批] 是否批准执行 '{action_name}'？(y/n): ")
    
    if approval.lower() == 'y':
        print("[审批] 已批准")
        return action_func(**kwargs)
    else:
        print("[审批] 已拒绝")
        return {"status": "rejected", "reason": "人工拒绝"}

# 测试（自动审批模式）
print("=== 模式1：审批确认 ===")
result = execute_with_approval(
    "发送邮件",
    send_email,
    auto_approve=True,
    to="user@example.com",
    subject="测试",
    body="这是一封测试邮件"
)
print(f"结果: {result}")

=== 模式1：审批确认 ===

[系统] 准备执行: 发送邮件
[系统] 参数: {'to': 'user@example.com', 'subject': '测试', 'body': '这是一封测试邮件'}
[系统] 自动审批通过
结果: {'status': 'sent', 'to': 'user@example.com', 'subject': '测试'}


## 模式2：内容审核

人类审核 LLM 生成的内容

In [4]:
# 生成内容
generate_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个营销文案 writer。请写一段产品描述。"),
    ("user", "产品: {product}")
])

generate_chain = generate_prompt | llm | StrOutputParser()

# 审核内容
review_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个内容审核员。请检查内容：
1. 是否有不当内容
2. 是否有事实错误
3. 是否符合品牌调性

输出：通过 / 需修改：原因"""),
    ("user", "内容: {content}")
])

review_chain = review_prompt | llm | StrOutputParser()

# 带审核的内容生成
def generate_with_review(product: str, auto_approve: bool = False):
    """生成内容并审核"""
    # 生成内容
    content = generate_chain.invoke({"product": product})
    print(f"\n[生成内容]\n{content}")
    
    # 审核
    review = review_chain.invoke({"content": content})
    print(f"\n[审核结果]\n{review}")
    
    if "通过" in review:
        print("\n[系统] 内容审核通过")
        return content
    else:
        print("\n[系统] 内容需要修改")
        if auto_approve:
            print("[系统] 自动使用（开发模式）")
            return content
        return None

# 测试
print("=== 模式2：内容审核 ===")
result = generate_with_review("小米智能手环", auto_approve=True)
if result:
    print(f"\n[最终内容]\n{result[:100]}...")

=== 模式2：内容审核 ===



[生成内容]
# 小米智能手环 — 你的腕上健康管家

---

**轻盈随行，智在掌握。**

每一天，从手腕开始，读懂自己的身体。

---

## ✨ 为什么选择小米智能手环？

🔹 **全天候健康监测**
24小时实时心率追踪、血氧饱和度检测、睡眠质量分析，让你对自己的身体了如指掌。每一次心跳，都被温柔记录。

🔹 **专业运动模式**
支持跑步、骑行、游泳等100+种运动模式，精准记录卡路里消耗、运动轨迹和训练效果，让每一滴汗水都有价值。

🔹 **超长续航，告别焦虑**
一次充电，续航长达14天。轻装出行，无需为电量担忧。

🔹 **轻若无物的佩戴体验**
仅重约16g，1.62英寸高清AMOLED大屏，佩戴舒适无感，时尚百搭，无论运动还是日常通勤，都是你的最佳配饰。

🔹 **智能生活小助手**
来电提醒、消息通知、天气预报、闹钟、久坐提醒……抬手即知，生活效率瞬间拉满。

🔹 **5ATM专业防水**
游泳、淋雨、洗手，无需摘下，陪你征服每一个场景。

---

## 💬 用户真实感受

> *"戴了两周，睡眠质量真的改善了，才知道自己以前睡得有多差！"*
> *"性价比之王，功能比很多千元手表还全面。"*
> *"太轻了，有时候都忘了自己戴着它。"*

---

## 🎯 适合谁？

✅ 想要科学管理健康的你
✅ 热爱运动、追求数据的你
✅ 追求高性价比智能穿戴的你
✅ 忙碌但不想错过重要消息的你

---

**小米智能手环 — 不只是一块手环，更是你健康生活的第一步。**

🛒 **现在入手，开启你的智能健康新生活！**

> *科技，让生活更美好。*



[审核结果]
通过

[系统] 内容审核通过

[最终内容]
# 小米智能手环 — 你的腕上健康管家

---

**轻盈随行，智在掌握。**

每一天，从手腕开始，读懂自己的身体。

---

## ✨ 为什么选择小米智能手环？

🔹 **全天候健康监测**
...


## 模式3：升级处理

复杂问题自动转交人类处理

In [5]:
# 客服分类
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个客服分类器。判断问题类型：
- 简单问题：可自动回答
- 复杂问题：需要人工处理
- 投诉：需要升级处理

输出：简单问题 / 复杂问题 / 投诉"""),
    ("user", "问题: {question}")
])

classify_chain = classify_prompt | llm | StrOutputParser()

# 自动回答
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个客服助手。请简洁回答用户问题。"),
    ("user", "{question}")
])

answer_chain = answer_prompt | llm | StrOutputParser()

# 升级到人工
def escalate_to_human(question: str, reason: str) -> dict:
    """模拟转交人工处理"""
    return {
        "status": "escalated",
        "question": question,
        "reason": reason,
        "message": "您的问题已转交人工客服，请稍候。"
    }

# 带升级的客服系统
def customer_support(question: str, auto_approve: bool = False):
    """带升级处理的客服系统"""
    # 分类
    category = classify_chain.invoke({"question": question})
    print(f"\n[分类] {category}")
    
    if "简单" in category:
        # 自动回答
        answer = answer_chain.invoke({"question": question})
        print(f"[自动回答] {answer}")
        return {"type": "auto", "answer": answer}
    
    elif "复杂" in category or "投诉" in category:
        # 升级到人工
        if auto_approve:
            result = escalate_to_human(question, category)
            print(f"[升级] {result['message']}")
            return result
        else:
            print(f"[系统] 需要人工处理，是否升级？")
            return {"type": "needs_human", "category": category}
    
    return {"type": "unknown"}

# 测试
print("=== 模式3：升级处理 ===")

print("\n--- 简单问题 ---")
customer_support("你们的营业时间是什么？", auto_approve=True)

print("\n--- 投诉问题 ---")
customer_support("我买的产品质量太差了，要求退款！", auto_approve=True)

=== 模式3：升级处理 ===

--- 简单问题 ---



[分类] 简单问题


[自动回答] 我们的营业时间为：  
**周一至周五**：9:00 - 18:00  
**周六、周日**：10:00 - 16:00  
节假日可能调整，建议提前确认哦~ 😊  
有其他问题随时问我！

--- 投诉问题 ---



[分类] 投诉
[升级] 您的问题已转交人工客服，请稍候。


{'status': 'escalated',
 'question': '我买的产品质量太差了，要求退款！',
 'reason': '投诉',
 'message': '您的问题已转交人工客服，请稍候。'}

## 完整的人机协作客服系统

组合三种模式：分类 → 自动/人工 → 审核 → 执行

In [6]:
def full_customer_support(question: str, auto_approve: bool = False):
    """完整的人机协作客服系统"""
    print(f"\n{'='*50}")
    print(f"用户问题: {question}")
    print(f"{'='*50}")
    
    # Step 1: 分类
    category = classify_chain.invoke({"question": question})
    print(f"\n[Step 1] 分类: {category}")
    
    # Step 2: 根据分类处理
    if "简单" in category:
        # 自动回答
        answer = answer_chain.invoke({"question": question})
        print(f"\n[Step 2] 自动回答:")
        print(answer)
        
        # Step 3: 审核回答
        review = review_chain.invoke({"content": answer})
        print(f"\n[Step 3] 审核: {review}")
        
        if "通过" in review:
            print("\n[最终] 回答已发送")
            return {"status": "sent", "answer": answer}
        else:
            print("\n[最终] 回答需要修改")
            return {"status": "needs_revision", "answer": answer}
    
    else:
        # 升级到人工
        result = escalate_to_human(question, category)
        print(f"\n[Step 2] 已升级: {result['message']}")
        return result

# 测试
print("\n" + "#"*50)
print("完整人机协作客服系统测试")
print("#"*50)

full_customer_support("你们支持哪些支付方式？", auto_approve=True)
print("\n" + "-"*50)
full_customer_support("我要投诉！产品有严重质量问题！", auto_approve=True)


##################################################
完整人机协作客服系统测试
##################################################

用户问题: 你们支持哪些支付方式？



[Step 1] 分类: 简单问题



[Step 2] 自动回答:
我们支持以下支付方式：

1. **支付宝**  
2. **微信支付**  
3. **银联卡**（借记卡/信用卡）  
4. **国际信用卡**（Visa、Mastercard等）  
5. **其他本地支付方式**（如云闪付等，具体以结算页面显示为准）

如有特殊需求，可联系客服进一步协助。



[Step 3] 审核: 通过

[最终] 回答已发送

--------------------------------------------------

用户问题: 我要投诉！产品有严重质量问题！



[Step 1] 分类: 投诉

[Step 2] 已升级: 您的问题已转交人工客服，请稍候。


{'status': 'escalated',
 'question': '我要投诉！产品有严重质量问题！',
 'reason': '投诉',
 'message': '您的问题已转交人工客服，请稍候。'}

## 人机协作流程

```
┌─────────────────────────────────────────────────────────┐
│                人机协作流程                               │
├─────────────────────────────────────────────────────────┤
│  用户请求                                                │
│    ↓                                                    │
│  LLM 分类（简单/复杂/投诉）                               │
│    ↓                                                    │
│  ┌─────────────────────────────────────┐                │
│  │ 简单 → LLM 自动生成回答             │                │
│  │ 复杂 → 升级到人工处理               │                │
│  │ 投诉 → 升级到人工处理               │                │
│  └─────────────────────────────────────┘                │
│    ↓                                                    │
│  人类审核（可选）                                         │
│    ↓                                                    │
│  执行 / 返回修改                                          │
└─────────────────────────────────────────────────────────┘
```

## 三种模式对比

| 模式 | 触发条件 | 人类参与点 | 适用场景 |
|------|----------|------------|----------|
| **审批确认** | 关键操作前 | 执行前审批 | 发邮件、删数据 |
| **内容审核** | 生成内容后 | 审核内容质量 | 文案、代码 |
| **升级处理** | 问题复杂时 | 接管处理 | 客服、决策 |

## 实际应用场景

- **客服系统**：简单问题自动回答，复杂问题转人工
- **内容生成**：AI 生成，人类审核后发布
- **代码审查**：AI 写代码，人类 review 后合并
- **决策支持**：AI 分析数据，人类做最终决策